In [3]:
import joblib
import pandas as pd

# Load model + features
model = joblib.load("ml_models/model1.pkl")
symptom_list = joblib.load("ml_models/symptoms.pkl")

# Load severity dataset
severity_df = pd.read_csv("data\Symptom-severity.csv")

severity_df["Symptom"] = (
    severity_df["Symptom"]
    .str.lower()
    .str.strip()
    .str.replace("_", " ")
)

severity_dict = dict(zip(severity_df["Symptom"], severity_df["weight"]))

<>:9: SyntaxWarning: invalid escape sequence '\S'
<>:9: SyntaxWarning: invalid escape sequence '\S'
C:\Users\adity\AppData\Local\Temp\ipykernel_11880\3114784722.py:9: SyntaxWarning: invalid escape sequence '\S'
  severity_df = pd.read_csv("data\Symptom-severity.csv")


In [4]:
def rule_based_checks(symptoms):
    symptoms = [s.lower() for s in symptoms]

    # 🚨 Critical rules
    if "chest pain" in symptoms and "breathlessness" in symptoms:
        return "⚠️ Possible heart-related emergency. Seek immediate medical help."

    if "high fever" in symptoms and "vomiting" in symptoms:
        return "⚠️ Possible severe infection. Consult a doctor."

    if "unconsciousness" in symptoms:
        return "🚨 Emergency condition. Immediate attention required."

    return None

In [5]:
def create_weighted_vector(input_symptoms):
    vector = [0] * len(symptom_list)

    for symptom in input_symptoms:
        symptom = symptom.lower().strip().replace("_", " ")

        if symptom in symptom_list:
            idx = symptom_list.index(symptom)

            # Apply severity weight
            weight = severity_dict.get(symptom, 1)

            vector[idx] = weight

    return vector

In [10]:
desc_df = pd.read_csv("data/symptom_Description.csv")

desc_df["Disease"] = desc_df["Disease"].str.lower().str.strip()
desc_dict = dict(zip(desc_df["Disease"], desc_df["Description"]))

In [12]:
prec_df = pd.read_csv("data/symptom_precaution.csv")

prec_df["Disease"] = prec_df["Disease"].str.lower().str.strip()

# Combine 4 precaution columns into list
prec_dict = {}

for _, row in prec_df.iterrows():
    precautions = [
        str(row[col]).strip()
        for col in prec_df.columns if "Precaution" in col and pd.notna(row[col])
    ]
    
    prec_dict[row["Disease"]] = precautions

In [13]:
def get_top_predictions(vector, top_n=3):
    probs = model.predict_proba([vector])[0]

    top_indices = probs.argsort()[-top_n:][::-1]

    results = []

    for idx in top_indices:
        disease = model.classes_[idx]

        results.append({
            "disease": disease,
            "confidence": round(probs[idx], 3),
            "description": desc_dict.get(disease, "No description available"),
            "precautions": prec_dict.get(disease, ["No precautions available"])
        })

    return results

In [14]:
def predict_system(input_symptoms):

    # 1. Rule-based check
    rule = rule_based_checks(input_symptoms)
    if rule:
        return {
            "type": "alert",
            "message": rule
        }

    # 2. Weighted vector
    vector = create_weighted_vector(input_symptoms)

    # 3. Predictions
    top_preds = get_top_predictions(vector)

    # 4. Confidence filter
    if top_preds[0]["confidence"] < 0.5:
        return {
            "type": "uncertain",
            "message": "Not enough information. Please provide more symptoms."
        }

    return {
        "type": "prediction",
        "results": top_preds
    }

In [15]:
predict_system(["fever", "cough", "headache"])

e:\FINAL\myenv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


{'type': 'prediction',
 'results': [{'disease': 'gerd',
   'confidence': np.float64(0.994),
   'description': 'Gastroesophageal reflux disease, or GERD, is a digestive disorder that affects the lower esophageal sphincter (LES), the ring of muscle between the esophagus and stomach. Many people, including pregnant women, suffer from heartburn or acid indigestion caused by GERD.',
   'precautions': ['avoid fatty spicy food',
    'avoid lying down after eating',
    'maintain healthy weight',
    'exercise']},
  {'disease': 'common cold',
   'confidence': np.float64(0.003),
   'description': "The common cold is a viral infection of your nose and throat (upper respiratory tract). It's usually harmless, although it might not feel that way. Many types of viruses can cause a common cold.",
   'precautions': ['drink vitamin c rich drinks',
    'take vapour',
    'avoid cold food',
    'keep fever in check']},
  {'disease': 'hypertension',
   'confidence': np.float64(0.001),
   'description': 'H